In [7]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio


In [2]:
from huggingface_hub import login

# You will be prompted to paste your HF token (make a token at: https://huggingface.co/settings/tokens)
login()


In [3]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# --------- MODEL LOADING ----------
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",           # Uses GPU if available
    torch_dtype=torch.float16,
)

model.eval()

LABELS = ["INFORMATIVE", "FIGURE_RELATED", "OTHER", "ASSUMPTION"]

# --------- FASTAPI APP ----------
app = FastAPI()

class PredictRequest(BaseModel):
    data: list   # [{"text": "..."}]


def classify(text: str) -> str:
    """Prompt-based classification using the 7B model"""
    prompt = f"""
You are a classification engine. Categorize the patent sentence into EXACTLY ONE of the labels below.

LABEL DEFINITIONS:
- INFORMATIVE: Technical, structural, functional, or operational details that describe HOW the invention works or WHAT it consists of.
- FIGURE_RELATED: Text that refers to drawings, figures, diagrams, or numbered illustrations (e.g., "FIG. 2", "as shown in the figure").
- OTHER: Boilerplate, legalistic, generic, redundant, or non-technical description that does NOT add technical value (e.g., "In another embodiment", "The invention relates to", background filler).
- ASSUMPTION: Statements making hypothetical, speculative, or optional claims, preferences, or assumptions without adding firm technical detail (e.g., "may be", "could be", "preferably", "in some embodiments").

TASK:
Classify the following patent sentence into exactly ONE category.

Sentence:
"{text}"

Return ONLY the label name with no punctuation, explanation, or extra words.
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=5,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(output[0], skip_special_tokens=True).strip()

    for label in LABELS:
        if label.lower() in answer.lower():
            return label

    return "OTHER"  # fallback


@app.get("/health")
def health():
    return {"status": "ok"}


# 👇 replace old @app.get("/setup") with this:
@app.api_route("/setup", methods=["GET", "POST"])
def setup():
    return {
        "from_name": "sentiment",   # 👈 control name in LS config
        "to_name": "text",          # 👈 data object name in LS config
        "type": "choices",
        "labels": ["INFORMATIVE", "FIGURE_RELATED", "OTHER", "ASSUMPTION"],
    }


@app.post("/predict")
def predict(request: PredictRequest):
    results = []
    for item in request.data:
        text = item["text"]
        pred = classify(text)

        results.append({
            "result": [
                {
                    "from_name": "sentiment",  # 👈 match LS config
                    "to_name": "text",         # 👈 match LS config
                    "type": "choices",
                    "value": {"choices": [pred]},
                }
            ],
            "score": 1.0,
        })
    return results



Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

In [4]:
import uvicorn, threading, nest_asyncio, time, requests

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)  # give it a few seconds to start

# quick test: local health
print(requests.get("http://127.0.0.1:8000/health").json())


INFO:     Started server process [1676]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:33268 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok'}


In [5]:
from google.colab import output

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
public_url


'https://8000-gpu-a100-s-1n4vj3bpvxvks-c.asia-southeast1-0.prod.colab.dev'

In [6]:
from pyngrok import ngrok

# kill old tunnels in this session
ngrok.kill()

# your auth token
ngrok.set_auth_token("2PgsprcdKolcczw6ru6HXbLcYfC_7cUSXnTdho7wqZyHYotoF")

# A) random domain
# public_url = ngrok.connect(addr="127.0.0.1:8000")

# B) your reserved free domain
public_url = ngrok.connect(
    addr="127.0.0.1:8000",
    domain="empiristic-mariyah-unprophetically.ngrok-free.dev"
)

print("Public URL:", public_url)


Public URL: NgrokTunnel: "https://empiristic-mariyah-unprophetically.ngrok-free.dev" -> "http://127.0.0.1:8000"


InvalidSchema: No connection adapters were found for '127.0.0.1:8000/api/predictions/bulk/'